In [1]:
from fileformer import utils, Decoder, Encoder

In [2]:
import torch
from flax import nnx
from torchinfo import summary

In [3]:
decoder = Decoder(258, 768, 6, 4, 'cpu', 768, 0.0, 256, 'gelu')

In [4]:
summary(decoder)

Layer (type:depth-idx)                             Param #
Decoder                                            --
├─Embedding: 1-1                                   198,144
├─RotaryPositionalEmbeddings: 1-2                  --
├─ModuleList: 1-3                                  --
│    └─DecoderLayer: 2-1                           --
│    │    └─MultiHeadLinearAttention: 3-1          2,362,368
│    │    └─MLP: 3-2                               1,181,184
│    │    └─LayerNorm: 3-3                         1,536
│    │    └─LayerNorm: 3-4                         1,536
│    │    └─Dropout: 3-5                           --
│    └─DecoderLayer: 2-2                           --
│    │    └─MultiHeadLinearAttention: 3-6          2,362,368
│    │    └─MLP: 3-7                               1,181,184
│    │    └─LayerNorm: 3-8                         1,536
│    │    └─LayerNorm: 3-9                         1,536
│    │    └─Dropout: 3-10                          --
│    └─DecoderLayer: 2-3        

In [5]:
from fileformer import ENWIK8Dataset
from fileformer.tokenizer import ByteLevelTokenizer
import torch
from torch import Tensor
from torch.utils.data import DataLoader

In [17]:
dataset = ENWIK8Dataset('enwik8', ByteLevelTokenizer(), 16256, 256, "cache")
loader = DataLoader(dataset, batch_size=2, shuffle=False)

In [21]:
x = next(iter(loader))

In [22]:
x.to(torch.long)

tensor([[ 62, 111, 103,  ..., 103, 111,  34],
        [123,  95,  40,  ...,  69, 106, 116]])

In [23]:
decoder.eval()

Decoder(
  (chunk_emb): Embedding(258, 768)
  (pe): RotaryPositionalEmbeddings()
  (layers): ModuleList(
    (0-3): 4 x DecoderLayer(
      (self_attention): MultiHeadLinearAttention(
        (Q_layer): Linear(in_features=768, out_features=768, bias=True)
        (K_layer): Linear(in_features=768, out_features=768, bias=True)
        (V_layer): Linear(in_features=768, out_features=768, bias=True)
        (fc_out): Linear(in_features=768, out_features=768, bias=True)
      )
      (mlp): MLP(
        (activation): GELU(approximate='none')
        (mlp): Sequential(
          (0): Linear(in_features=768, out_features=768, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=768, out_features=768, bias=True)
          (3): Dropout(p=0.0, inplace=False)
        )
      )
      (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (norm3): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )

In [24]:
out = decoder(x)

In [25]:
out.shape

torch.Size([2, 16256, 258])